# 🔬 Notebook 3 — Deploy Strategies & SLO-gated Auto-Rollback

You have an immutable artifact (Notebook 2). Now you have to get it onto running servers **without breaking users**. This notebook walks through the three classic strategies — **rolling**, **blue/green**, **canary** — from worst to best, and then shows a tiny **automatic rollback** driven by error-rate metrics.

## 🛠️ Setup

```bash
cd 06-system-designs/code-deployment
uv sync
```

Pick the `.venv` kernel. Reload the window if needed.

## 1. The bad baseline — "big-bang" deploy

Stop *all* old instances, then start *all* new ones. Simple. Also: 100% downtime + no easy rollback.

In [1]:
# 🔴 BAD: big-bang. Zero overlap ⇒ full outage window.
import time

class BigBang:
    def __init__(self, n=4):
        self.fleet = [{"id": i, "version": "v1", "healthy": True} for i in range(n)]

    def deploy(self, new_version: str):
        print("stopping ALL old instances…")
        for s in self.fleet:
            s["healthy"] = False
        time.sleep(0.1)
        print("⚠️  user traffic is 100% erroring right now")
        print("starting ALL new instances…")
        for s in self.fleet:
            s["version"] = new_version
            s["healthy"] = True
        print("done:", self.fleet)

BigBang().deploy("v2")

stopping ALL old instances…


⚠️  user traffic is 100% erroring right now
starting ALL new instances…
done: [{'id': 0, 'version': 'v2', 'healthy': True}, {'id': 1, 'version': 'v2', 'healthy': True}, {'id': 2, 'version': 'v2', 'healthy': True}, {'id': 3, 'version': 'v2', 'healthy': True}]


**Don't do this in production.** Use one of the three strategies below.

## 2. Rolling deploy — replace a few at a time

- Replace **N** instances at a time, wait for them to report healthy, then move on.
- At any moment, most of the fleet still serves traffic.
- Rollback = run a rolling deploy back to the previous version (takes as long as the forward deploy).

In [2]:
# 🟡 ROLLING: batch-by-batch with health checks
from dataclasses import dataclass

@dataclass
class Instance:
    id: int
    version: str
    healthy: bool = True

def health_check(inst: Instance) -> bool:
    # Simulate a probe. In real life: HTTP /healthz, startup time, warm-up, etc.
    time.sleep(0.02)
    return inst.healthy

def rolling(fleet: list[Instance], new_version: str, batch: int = 2) -> bool:
    for i in range(0, len(fleet), batch):
        window = fleet[i:i + batch]
        print(f"replacing {[w.id for w in window]} → {new_version}")
        for w in window:
            w.healthy = False                    # drained from LB
        for w in window:
            w.version = new_version
            w.healthy = True                      # starts up
            if not health_check(w):
                print(f"❌ {w.id} failed health check — aborting")
                return False
        print(f"  fleet: {[(w.id, w.version) for w in fleet]}")
    return True

fleet = [Instance(i, "v1") for i in range(6)]
rolling(fleet, "v2", batch=2)

replacing [0, 1] → v2


  fleet: [(0, 'v2'), (1, 'v2'), (2, 'v1'), (3, 'v1'), (4, 'v1'), (5, 'v1')]
replacing [2, 3] → v2


  fleet: [(0, 'v2'), (1, 'v2'), (2, 'v2'), (3, 'v2'), (4, 'v1'), (5, 'v1')]
replacing [4, 5] → v2


  fleet: [(0, 'v2'), (1, 'v2'), (2, 'v2'), (3, 'v2'), (4, 'v2'), (5, 'v2')]


True

**Pros:** simple, no extra capacity needed.
**Cons:** during the rollout, some users hit `v1` and some hit `v2` — your code must tolerate that (schema changes, API changes, …). Rollback is *not* instant.

## 3. Blue/green — two fleets, flip the switch

Keep the old fleet (`blue`) running. Stand up a *whole* new fleet (`green`) with the new version. Run smoke tests against green. Once happy, **atomically switch the load balancer** to point at green. Rollback is *instant* — just flip back.

In [3]:
# 🟢 BLUE/GREEN
class BlueGreen:
    def __init__(self):
        self.blue  = [Instance(i, "v1") for i in range(4)]
        self.green: list[Instance] = []
        self.live  = "blue"

    def deploy(self, new_version: str) -> bool:
        print(f"standing up green @ {new_version}…")
        self.green = [Instance(100 + i, new_version) for i in range(4)]
        if not all(health_check(g) for g in self.green):
            print("❌ green unhealthy; keeping blue")
            return False
        print("🔀 flipping LB: blue → green")
        self.live = "green"
        # Keep blue around for N minutes so rollback is just flipping back.
        return True

    def rollback(self):
        print("🔁 flipping LB: green → blue (instant)")
        self.live = "blue"

bg = BlueGreen()
bg.deploy("v2")
print("live:", bg.live)
bg.rollback()
print("live:", bg.live)

standing up green @ v2…


🔀 flipping LB: blue → green
live: green
🔁 flipping LB: green → blue (instant)
live: blue


**Pros:** instant rollback, easy smoke testing against green before cutover.
**Cons:** 2× capacity during the deploy; stateful services need careful handling (DB migrations!).

## 4. Canary — let 1% of users test production for you

Start the new version alongside the old one and **shift a tiny fraction of traffic** to it. If metrics (error rate, latency, business KPI) stay healthy, ramp to 5%, 25%, 100%. Otherwise, roll back — only ~1% of users were exposed.

```
   100% v1  ──▶  99% v1 / 1% v2  ──▶  90/10  ──▶  50/50  ──▶  100% v2
                        ▲ bad metrics here ⇒ rollback, impact was 1%
```

In [4]:
# 🔵 CANARY routing: decide which version a given request goes to
import random

def route(pct_v2: float) -> str:
    return "v2" if random.random() * 100 < pct_v2 else "v1"

random.seed(42)
counts = {"v1": 0, "v2": 0}
for _ in range(10_000):
    counts[route(1.0)] += 1         # 1% canary
print("1% canary split over 10k requests:", counts)

1% canary split over 10k requests: {'v1': 9896, 'v2': 104}


### 4.1 A realistic canary runner with SLO gate

Strategy: ramp through `[1, 5, 25, 50, 100] %`. After each step, measure the canary's error rate and compare to baseline. If it's worse than baseline by > `threshold`, **roll back automatically**.

In [5]:
def simulate_requests(n: int, err_rate: float) -> list[bool]:
    """Return n booleans: True = error, False = success."""
    return [random.random() < err_rate for _ in range(n)]

def observed_error_rate(results: list[bool]) -> float:
    return (sum(results) / len(results)) if results else 0.0

def should_rollback(baseline_err: float, canary_err: float,
                    threshold: float = 0.5, min_requests: int = 500,
                    canary_n: int = 0, baseline_n: int = 0) -> bool:
    """
    Rollback if canary error rate is worse than baseline by > threshold (relative),
    AND we have enough samples on *both* sides to trust the comparison.
    """
    if canary_n < min_requests or baseline_n < min_requests:
        return False                          # not enough data yet
    return canary_err > baseline_err * (1 + threshold)

def canary_deploy(baseline_err: float, real_canary_err: float,
                  steps=(1, 5, 25, 50, 100), traffic_per_step: int = 20000) -> str:
    print(f"baseline err={baseline_err:.2%}  canary err={real_canary_err:.2%}")
    for pct in steps:
        canary_reqs   = int(traffic_per_step * pct / 100)
        baseline_reqs = traffic_per_step - canary_reqs
        c_results = simulate_requests(canary_reqs,   real_canary_err)
        b_results = simulate_requests(baseline_reqs, baseline_err)
        c_err = observed_error_rate(c_results)
        b_err = observed_error_rate(b_results)
        print(f"  step {pct:>3}%  canary_reqs={canary_reqs:<5}  "
              f"b_err={b_err:.2%}  c_err={c_err:.2%}")
        if should_rollback(b_err, c_err, threshold=0.5,
                           canary_n=canary_reqs, baseline_n=baseline_reqs):
            print(f"  🚨 rollback! canary err ({c_err:.2%}) > baseline ({b_err:.2%}) × 1.5")
            return "rolled_back"
    return "promoted"

# Case 1: canary is fine (roughly the same error rate as baseline)
random.seed(1)
print("— healthy canary —")
print("result:", canary_deploy(baseline_err=0.01, real_canary_err=0.012))

print()

# Case 2: canary is broken (5× worse)
random.seed(2)
print("— broken canary —")
print("result:", canary_deploy(baseline_err=0.01, real_canary_err=0.05))

— healthy canary —
baseline err=1.00%  canary err=1.20%
  step   1%  canary_reqs=200    b_err=1.12%  c_err=1.50%
  step   5%  canary_reqs=1000   b_err=1.08%  c_err=1.60%
  step  25%  canary_reqs=5000   b_err=0.98%  c_err=1.04%
  step  50%  canary_reqs=10000  b_err=0.94%  c_err=1.25%
  step 100%  canary_reqs=20000  b_err=0.00%  c_err=1.11%
result: promoted

— broken canary —
baseline err=1.00%  canary err=5.00%
  step   1%  canary_reqs=200    b_err=1.04%  c_err=6.50%
  step   5%  canary_reqs=1000   b_err=0.99%  c_err=5.00%
  🚨 rollback! canary err (5.00%) > baseline (0.99%) × 1.5
result: rolled_back


### 4.2 What makes a good SLO gate?

- **Use a relative threshold.** `canary_err > baseline_err × 1.5` adapts to noisy services.
- **Require a minimum sample size.** Otherwise 1 error out of 10 requests = 10% error and you rollback for nothing.
- **Compare to the *current* baseline**, not yesterday's — if the whole internet is having a bad day, your canary will look bad too, but so does baseline, so relative comparison stays fair.
- **Watch multiple signals.** Error rate *and* p99 latency *and* a business KPI (e.g. checkout success). A regression in any one triggers rollback.
- **Bake time.** Hold each step for *minutes*, not seconds, so slow-burning bugs have time to show up.

## 5. Pulling it together — which strategy when?

| Strategy | Downtime | Extra capacity | Rollback speed | Good for |
|---|---|---|---|---|
| Big-bang | 100% window | 0 | slow | 🚫 never (except toys) |
| Rolling  | 0 | 0 | slow (full rolldown) | stateless services, backward-compatible changes |
| Blue/green | 0 | 2× | **instant flip** | risky releases; easy smoke-test gate |
| Canary   | 0 | small | fast (small blast radius) | user-facing services with good metrics |

### Bonus concepts (not coded, but important)

- **Feature flags.** Deploy the code dark, then flip a flag to enable it for 1% → 100% of users. Decouples *deploy* from *release*.
- **Progressive delivery.** Canary + automated metric analysis (e.g. Argo Rollouts, Flagger).
- **Forward-compatible DB migrations.** Always ship a migration that the old *and* new code can run against, then clean up later. Without this, rollback is dangerous.
- **Deployment windows & freezes.** Friday-afternoon prod deploys are where legends retire early.

## 🔑 Key takeaways

1. **Never big-bang.** Pick rolling, blue/green, or canary based on risk and capacity.
2. **Canary + automated rollback** gives you the smallest blast radius per risky change.
3. Rollback safety comes from **immutable artifacts** (Notebook 2) + a way to **shift traffic** (this notebook).
4. Deploy ≠ release. **Feature flags** let you ship code dark and release it separately.

Congrats — you now have the mental model (and runnable snippets) of a real-world code-deployment system. 🚀